# Dual Stratification Dataset: Post-1990 Democratizers Panel (1990-2024)

## Dataset Overview

This notebook demonstrates the **Dual Stratification Dataset** that combines:
- V-Dem Liberal Democracy Index (v2x_libdem)
- V-Dem Political Equality Index (v2pepwrsoc)  
- World Bank Gini coefficient (income inequality)
- Education spending as % of GDP
- Tertiary enrollment rates

**Research Question**: How do inequality, education, and democratic quality co-evolve across post-1990 democratizers?

**Dataset Stats**:
- 1,291 country-year observations  
- 38 countries (1990-2023)
- 4 true post-1990 democratizers: Bulgaria, Cape Verde, Latvia, Namibia
- 94.7% complete cases


In [ ]:
# Install dependencies - Colab compatible
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2')


In [ ]:
# Imports
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
print('All imports successful')


In [ ]:
# Data loading with GitHub URL (for Colab) and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-27ddf1-inequality-political-equality-and-democr/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        print('Loading from GitHub...')
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f'GitHub failed: {e}')
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

print('Load function defined')


In [ ]:
# Load the demo data
data = load_data()
examples = data['datasets'][0]['examples']
print(f'Loaded {len(examples)} examples')


## Understanding the Data Structure

The dataset is formatted as **examples** with:
- `input`: JSON string of feature values
- `output`: v2x_libdem value (Liberal Democracy Index, 0-1)
- `metadata_country`, `metadata_year`, etc.


In [ ]:
# Parse examples into DataFrame
rows = []
for ex in examples:
    row = json.loads(ex['input'])
    row['v2x_libdem'] = float(ex['output'])
    row['country'] = ex['metadata_country']
    row['year'] = ex['metadata_year']
    row['post_1990_democratizer'] = ex['metadata_post_1990_democratizer']
    rows.append(row)

df = pd.DataFrame(rows)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(df.head())


## Data Exploration

In [ ]:
# Descriptive statistics
print('Countries:', sorted(df['country'].unique()))
print(f'Year range: {df["year"].min()}-{df["year"].max()}')
print('\nDemocratizers:', df[df['post_1990_democratizer']]['country'].unique().tolist())
print('\n=== Numeric Summary ===')
print(df[['gini', 'education_spending_gdp', 'v2x_libdem']].describe().round(3))


## Visualizations

In [ ]:
# Plot 1: Democracy over time by country
plt.figure(figsize=(12, 6))
for country in df['country'].unique():
    cd = df[df['country'] == country].sort_values('year')
    ls = '-' if cd['post_1990_democratizer'].iloc[0] else '--'
    plt.plot(cd['year'], cd['v2x_libdem'], label=country, linestyle=ls, alpha=0.7)
plt.xlabel('Year')
plt.ylabel('Liberal Democracy Index')
plt.title('Democracy Trajectories')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Gini vs Democracy
plt.figure(figsize=(10, 6))
d = df[df['post_1990_democratizer'] == True]
e = df[df['post_1990_democratizer'] == False]
plt.scatter(e['gini'], e['v2x_libdem'], alpha=0.6, label='Established')
plt.scatter(d['gini'], d['v2x_libdem'], alpha=0.8, label='Democratizers', s=80)
plt.xlabel('Gini Coefficient')
plt.ylabel('Democracy Index')
plt.title('Inequality vs Democratic Quality')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Correlation: {df["gini"].corr(df["v2x_libdem"]):.3f}')


In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
cols = ['gini', 'education_spending_gdp', 'tertiary_enrollment', 'v2pepwrsoc', 'v2x_libdem']
sns.heatmap(df[cols].corr(), annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


## Comparison: Democratizers vs Established Democracies

In [ ]:
# Compare means
d = df[df['post_1990_democratizer'] == True]
e = df[df['post_1990_democratizer'] == False]
comparison = pd.DataFrame({
    'Democratizers': d[cols].mean(),
    'Established': e[cols].mean()
})
print(comparison.round(3))


## Key Findings

1. **Democracy Trajectories**: Post-1990 democratizers show different patterns.
2. **Inequality**: Relationship between Gini and democracy revealed.
3. **Education**: Correlations with democratic quality shown.

## Next Steps
- Use full dataset (1,291 observations) for regression with fixed effects
- Test if education/welfare mediates inequality-democracy relationship
- Analyze transitional dynamics during democratic transitions
